<a href="https://colab.research.google.com/github/PauloCunhaJunior/criar-imagens/blob/main/imagem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reconhecimento de Imagens com CNN e TensorFlow usando MNIST

Este algoritmo tem como objetivo treinar uma rede neural convolucional, conhecida como CNN, para reconhecer imagens de números escritos de 0 a 9.

O projeto utiliza o conjunto de dados MNIST, que contém várias imagens de dígitos manuscritos. Depois do treinamento, o algoritmo também cria novas imagens de números por código e testa se o modelo consegue reconhecê-las corretamente.

De forma geral, o algoritmo faz três coisas principais:

1. Treina uma rede neural convolucional para reconhecer números de 0 a 9.
2. Cria imagens novas com números desenhados por código.
3. Envia essas imagens criadas para o modelo treinado fazer a previsão.

Este código funciona bem no Google Colab, pois o ambiente já possui suporte melhor para bibliotecas como TensorFlow.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont
from tensorflow.keras import datasets, layers, models

In [ ]:
IMAGE_SIZE = 28
OUTPUT_DIR = Path("imagens_criadas")

In [ ]:
def treinar_modelo():
    """
    Carrega o conjunto de dados MNIST, prepara as imagens,
    cria a rede neural CNN, treina o modelo e devolve o modelo treinado.
    """

    # Carrega o conjunto de dados MNIST
    (train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

    # Normaliza os valores dos pixels para ficarem entre 0 e 1
    train_images = train_images / 255.0
    test_images = test_images / 255.0

    # Ajusta o formato das imagens para o padrão esperado pela CNN
    train_images = train_images.reshape((train_images.shape[0], 28, 28, 1))
    test_images = test_images.reshape((test_images.shape[0], 28, 28, 1))

    # Criação do modelo CNN
    model = models.Sequential(
        [
            layers.Input(shape=(28, 28, 1)),

            layers.Conv2D(32, (3, 3), activation="relu"),
            layers.MaxPooling2D((2, 2)),

            layers.Conv2D(64, (3, 3), activation="relu"),
            layers.MaxPooling2D((2, 2)),

            layers.Conv2D(64, (3, 3), activation="relu"),

            layers.Flatten(),
            layers.Dense(64, activation="relu"),
            layers.Dense(10, activation="softmax"),
        ]
    )

    # Compilação do modelo
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    # Treinamento do modelo
    model.fit(
        train_images,
        train_labels,
        epochs=5,
        batch_size=64,
        validation_data=(test_images, test_labels),
    )

    # Avaliação do modelo com os dados de teste
    test_loss, test_accuracy = model.evaluate(test_images, test_labels, verbose=0)

    print(f"Perda nos dados de teste: {test_loss:.4f}")
    print(f"Precisão nos dados de teste: {test_accuracy:.4f}")

    return model, test_images, test_labels

In [ ]:
def carregar_fonte(tamanho):
    """
    Tenta carregar uma fonte disponível no Colab ou no Windows.
    Caso não encontre, usa uma fonte padrão.
    """

    fontes = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "C:/Windows/Fonts/arialbd.ttf",
        "arialbd.ttf",
        "arial.ttf",
    ]

    for fonte in fontes:
        try:
            return ImageFont.truetype(fonte, tamanho)
        except OSError:
            pass

    return ImageFont.load_default()

In [ ]:
def criar_imagem_digito(digito, caminho, tamanho_fonte=24, deslocamento=(0, -2)):
    """
    Cria uma imagem parecida com as imagens do MNIST.

    A imagem terá:
    - tamanho 28x28 pixels;
    - fundo preto;
    - número branco no centro.
    """

    imagem = Image.new("L", (IMAGE_SIZE, IMAGE_SIZE), color=0)
    desenho = ImageDraw.Draw(imagem)
    fonte = carregar_fonte(tamanho_fonte)

    texto = str(digito)

    caixa = desenho.textbbox((0, 0), texto, font=fonte)

    largura = caixa[2] - caixa[0]
    altura = caixa[3] - caixa[1]

    x = (IMAGE_SIZE - largura) // 2 + deslocamento[0]
    y = (IMAGE_SIZE - altura) // 2 + deslocamento[1]

    desenho.text((x, y), texto, fill=255, font=fonte)

    imagem.save(caminho)

    return caminho

In [ ]:
def preparar_imagem_para_modelo(caminho):
    """
    Abre uma imagem externa e transforma no formato esperado pela CNN.

    O modelo espera uma imagem:
    - em tons de cinza;
    - com tamanho 28x28;
    - com valores entre 0 e 1;
    - no formato (1, 28, 28, 1).
    """

    imagem = Image.open(caminho).convert("L").resize((IMAGE_SIZE, IMAGE_SIZE))

    array = np.array(imagem).astype("float32") / 255.0

    # Se a imagem tiver fundo claro e número escuro, inverte as cores
    if array.mean() > 0.5:
        array = 1.0 - array

    return array.reshape(1, IMAGE_SIZE, IMAGE_SIZE, 1)

In [ ]:
def avaliar_imagem(model, caminho):
    """
    Envia uma imagem para o modelo treinado e mostra qual número foi previsto.
    """

    imagem_modelo = preparar_imagem_para_modelo(caminho)

    probabilidades = model.predict(imagem_modelo, verbose=0)[0]

    previsao = int(np.argmax(probabilidades))
    confianca = float(probabilidades[previsao])

    plt.figure(figsize=(2, 2))
    plt.imshow(imagem_modelo.reshape(IMAGE_SIZE, IMAGE_SIZE), cmap="gray")
    plt.title(f"Previsto: {previsao} ({confianca:.1%})")
    plt.axis("off")
    plt.show()

    print(f"Arquivo: {caminho}")
    print(f"Número previsto: {previsao}")
    print(f"Confiança: {confianca:.1%}")
    print("-" * 30)

    return previsao, confianca

In [ ]:
def testar_com_imagens_do_mnist(model, test_images, test_labels, quantidade=10):
    """
    Testa o modelo usando imagens reais do conjunto MNIST.
    """

    imagens = test_images[:quantidade]
    rotulos = test_labels[:quantidade]

    previsoes = model.predict(imagens, verbose=0)

    rotulos_previstos = np.argmax(previsoes, axis=1)

    for i, imagem in enumerate(imagens):
        plt.figure(figsize=(1.5, 1.5))
        plt.imshow(imagem.reshape(28, 28), cmap="gray")
        plt.title(f"Real: {rotulos[i]} | Previsto: {rotulos_previstos[i]}")
        plt.axis("off")
        plt.show()

In [ ]:
def criar_e_avaliar_novas_imagens(model):
    """
    Cria uma imagem para cada número de 0 a 9
    e testa se o modelo consegue reconhecer cada uma.
    """

    OUTPUT_DIR.mkdir(exist_ok=True)

    for digito in range(10):
        caminho = OUTPUT_DIR / f"digito_{digito}.png"

        criar_imagem_digito(digito, caminho)

        avaliar_imagem(model, caminho)

In [ ]:
modelo, imagens_teste, rotulos_teste = treinar_modelo()

In [ ]:
print("Teste com imagens reais do MNIST:")

testar_com_imagens_do_mnist(modelo, imagens_teste, rotulos_teste)

In [ ]:
print("Teste com imagens criadas por código:")

criar_e_avaliar_novas_imagens(modelo)